In [1]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim

from transformer.vocabs import get_or_create_smiles_vocabs
from transformer.data_funcs import calculate_max_mz
from transformer.transformer_utils import split_and_load_tokenized_multimodal_data
from transformer.data_funcs import bin_spectrum
from transformer.models import MultimodalVITSeq2SeqBeam
from transformer.sequence_pred import train_multimodal_beam_model
from transformer.evaluation import evaluate_multimodal_beam_model, plot_training_history

In [2]:
if torch.cuda.is_available():
    print("CUDA is available.")
    print("PyTorch version:", torch.__version__)
    print("CUDA version:", torch.version.cuda)
    print("Number of available GPUs:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

CUDA is available.
PyTorch version: 2.1.0+cu121
CUDA version: 12.1
Number of available GPUs: 1
GPU name: NVIDIA GeForce RTX 4060 Laptop GPU


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
ir = pd.read_feather('data/ir_spectra_v2.feather')
ir.columns= ['SMILES', 'spectrum']

In [5]:
ir.head()

,SMILES,spectrum
0,COc1nc2ccccc2cc1C(=O)O,"[-0.004481, 0.007279, -0.006439, 0.007464, -0...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[0.028195, 0.011433, 0.024865, -0.00816, 0.055..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[-0.003043, 0.014693, -0.003067, 0.025262, -0...."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[0.099321, 0.155452, 0.048156, 0.19118, 0.0237..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[0.2335, 0.090925, 0.061724, 0.158822, 0.04248..."


In [6]:
ms = pd.read_feather('data/msms_cfmid_positive_40ev_v2.feather')
ms.columns = ['SMILES', 'spectrum']

In [7]:
ms.head()

,SMILES,spectrum
0,COc1nc2ccccc2cc1C(=O)O,"[[101.03858, 100.0], [103.05423, 43.72], [104...."
1,CCOC(=O)c1cc2c(OCc3coc4cc(F)ccc34)cccc2n1C(=O)...,"[[57.06988, 51.14], [91.05423, 9.59], [121.044..."
2,CCCCOc1c(CN2C(=O)c3ccccc3C2=O)n(CC2CC2)c(=O)c2...,"[[57.06988, 28.12], [105.03349, 100.0], [117.0..."
3,CCCCNC[C@@H]1O[C@](O)(CO)[C@@H](O)[C@@H]1O,"[[30.03383, 36.99], [41.03858, 8.43], [42.0338..."
4,O=C(NCC1CC1)c1nc2c(N3CCC(n4c(=O)[nH]c5ccccc54)...,"[[55.05423, 63.08], [70.02874, 18.67], [70.065..."


In [8]:
# Minimal test
n = 50000
df_ir = ir.sample(n, random_state=42)
df_ir_test = ir.drop(index=df_ir.index).sample(n//2, random_state=42)
df_ms = ms.loc[df_ir.index]
df_ms_test = ms.loc[df_ir_test.index]

In [9]:
df_ir.reset_index(drop=True, inplace=True)
df_ir_test.reset_index(drop=True, inplace=True)
df_ms.reset_index(drop=True, inplace=True)
df_ms_test.reset_index(drop=True, inplace=True)

In [10]:
method='direct'

In [11]:
smiles_vocabs = get_or_create_smiles_vocabs(pd.concat([ms, ir]), smiles_col='SMILES', source='msd')

Loading existing character vocabulary...
SMILES vocabulary size (character): 39
Loading existing atom_wise vocabulary...
SMILES vocabulary size (atom_wise): 14
Loading existing substructure vocabulary...
SMILES vocabulary size (substructure): 3767143


In [12]:
max_mz = calculate_max_mz(pd.concat([ms]), spectrum_column='spectrum', dtype=np.array)

In [13]:
max_ir = ir['spectrum'].apply(lambda x: len(x)).max()

In [14]:
max_smiles = pd.concat([ms, ir])['SMILES'].apply(lambda x: len(x)).max()

In [15]:
max_smiles

127

In [16]:
pd.concat([ms, ir]).loc[pd.concat([ms, ir])['SMILES'].apply(lambda x: len(x)) > 1000]

,SMILES,spectrum


In [17]:
train_data = {
    'MS': df_ms,
    'IR': df_ir
}
test_data = {
    'MS': df_ms_test,
    'IR': df_ir_test
}
tokenization_methods = {
    'MS': 'direct',
    'IR': 'direct'
}
max_values = {
    'MS': max_mz,
    'IR': max_ir,
    'SMILES': max_smiles,
}

In [18]:
df_ms['spectrum'] = df_ms['spectrum'].apply(lambda x: bin_spectrum(x, max_mz))
df_ms_test['spectrum'] = df_ms_test['spectrum'].apply(lambda x: bin_spectrum(x, max_mz))

In [19]:
results = {}
for key, value in tokenization_methods.items():
    print(f"\n{key} spectra tokenized with {value} tokenization")
print(f"\nSMILES tokenized with {'character'} tokenization")
smiles_vocab = smiles_vocabs['character']

train_loader, test_loader = split_and_load_tokenized_multimodal_data(
    train_data_dict=train_data,
    train_smiles=df_ms['SMILES'],
    test_data_dict=test_data,
    test_smiles=df_ms_test['SMILES'],
    tokenization_methods=tokenization_methods,
    smiles_vocab=smiles_vocabs['character'],
    max_values=max_values,
)

#num_classes = len(label_encoder.classes_)
smiles_vocab_size = len(smiles_vocab)

# sample batch used for input dimensions
sample_batch, target_batch = next(iter(train_loader))
for key in tokenization_methods.keys():
    print(key, "spectra shape:", sample_batch[key].shape)
print("SMILES shape:", target_batch.shape)
embed_depth = sample_batch['MS'].shape[3]


MS spectra tokenized with direct tokenization

IR spectra tokenized with direct tokenization

SMILES tokenized with character tokenization
MS spectra shape: torch.Size([32, 1, 68, 16])
IR spectra shape: torch.Size([32, 1, 113, 16])
SMILES shape: torch.Size([32, 86])


In [20]:
modality_configs = {
    'MS': {
        'embed_depth': 16,
        'max_length': sample_batch['MS'].shape[2],
    },
    'IR': {
        'embed_depth': 16,
        'max_length': sample_batch['IR'].shape[2],
    }
}

In [21]:
modality_configs

{'MS': {'embed_depth': 16, 'max_length': 68},
 'IR': {'embed_depth': 16, 'max_length': 113}}

In [22]:
model = MultimodalVITSeq2SeqBeam(
    smiles_vocab_size=len(smiles_vocab),
    modality_configs=modality_configs,
    d_model=64,             # 256
    nhead=4,                # 8
    num_layers=2,           # 6
    dim_feedforward=256,    # 2048
)

/home/kyle/anaconda3/envs/ml/lib/python3.11/site-packages/torch/nn/modules/transformer.py:282: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [23]:
model.state_dict()

OrderedDict([('modality_embeddings.MS.weight',
              tensor([[-0.0506, -0.0015, -0.0287,  ..., -0.0772, -0.0841, -0.0076],
                      [-0.0359, -0.0204, -0.0064,  ..., -0.0753, -0.0728,  0.0489],
                      [-0.0758, -0.0204,  0.0512,  ...,  0.0976, -0.0205,  0.0793],
                      ...,
                      [-0.0064,  0.0637, -0.0795,  ...,  0.0187, -0.0054, -0.0039],
                      [-0.0675, -0.0797,  0.0172,  ...,  0.0099,  0.0903, -0.0401],
                      [ 0.0428,  0.0112,  0.0665,  ..., -0.0891, -0.0982,  0.0054]])),
             ('modality_embeddings.MS.bias',
              tensor([-0.0636, -0.1398, -0.1819,  0.0191,  0.1635,  0.1638,  0.1796,  0.0017,
                      -0.1247, -0.2062,  0.1274,  0.0594,  0.1307,  0.1461,  0.0793, -0.0579,
                      -0.0996,  0.0470, -0.1723, -0.0237,  0.0287, -0.1370, -0.2093,  0.1297,
                       0.1350, -0.0257,  0.0571, -0.2240, -0.2020, -0.2096,  0.0229, -0.1487

In [24]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.CrossEntropyLoss()#ignore_index=smiles_vocab['<pad>'])

model, history = train_multimodal_beam_model(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    optimizer=optimizer,
    criterion=criterion,
    num_epochs=1,
    evaluate=False,
    verbose=2,
    checkpoint_path='model_checkpoints/',
    use_tensorboard=True,
)

TensorBoard logs will be saved to runs/20250123-093059
Saving checkpoints to model_checkpoints/checkpoint_3


Epoch 1/1 [Train]:   0%|          | 0/1563 [00:00<?, ?it/s]

In [21]:
results = evaluate_multimodal_beam_model(
    model=model, 
    test_loader=test_loader, 
    smiles_vocab=smiles_vocab, 
    beam_width=5,
    verbose=1
)

Evaluating:   0%|          | 0/782 [00:00<?, ?it/s]

AttributeError: 'dict' object has no attribute 'size'

In [ ]:
plot_training_history(history)

In [ ]:
results

In [ ]:
history